# 4. Review the results and check for data shift

The model produces a ranked list, not a fraud probability. This notebook summarises the later-period scores and compares the feature distributions in the training and holdout periods.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance
from IPython.display import display

PROJECT = Path.cwd().resolve()
if not (PROJECT / "data").exists():
    PROJECT = PROJECT.parent
FEATURE_FILE = PROJECT / "data" / "processed" / "ppip_features.csv"
SCORES_FILE = PROJECT / "reports" / "holdout_scores.csv"
REPORT_DIR = PROJECT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
features = pd.read_csv(FEATURE_FILE, parse_dates=["tender_start_date"])
scores = pd.read_csv(SCORES_FILE, parse_dates=["tender_start_date"])
features = features.sort_values(["tender_start_date", "ocid"]).reset_index(drop=True)
split_date = features["tender_start_date"].quantile(0.70)
train = features[features["tender_start_date"] <= split_date].copy()
holdout = features[features["tender_start_date"] > split_date].copy()

## Ranking summary

The 90th and 98th training-percentile cut-offs are review bands. They do not have a known false-positive rate because the source does not contain verified outcomes.

In [ ]:
summary = pd.DataFrame([
    ("holdout_processes", len(scores)),
    ("high_band_count", int((scores["review_band"] == "high").sum())),
    ("review_band_count", int((scores["review_band"] == "review").sum())),
    ("routine_count", int((scores["review_band"] == "routine").sum())),
    ("median_anomaly_percentile", float(scores["anomaly_percentile"].median())),
    ("90th_percentile_of_holdout_scores", float(scores["anomaly_percentile"].quantile(0.90))),
    ("98th_percentile_of_holdout_scores", float(scores["anomaly_percentile"].quantile(0.98))),
], columns=["measure", "value"])
summary.to_csv(REPORT_DIR / "evaluation_summary.csv", index=False)
display(summary)
display(scores.head(20))

## Plot the score distribution

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(scores["anomaly_percentile"], bins=25)
plt.xlabel("Percentile relative to training anomaly scores")
plt.ylabel("Number of processes")
plt.title("Anomaly ranking in the later-period holdout")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "holdout_anomaly_percentiles.png", dpi=160)
plt.show()

## Compare the training and holdout feature distributions

Wasserstein distance is reported for numeric features. It measures a change in distribution; it does not say why the change happened or whether a tender is improper.

In [ ]:
excluded = {"ocid", "tender_start_date", "tender_text"}
numeric_features = [
    col for col in features.columns
    if col not in excluded and pd.api.types.is_numeric_dtype(features[col])
]
shift_rows = []
for col in numeric_features:
    before = pd.to_numeric(train[col], errors="coerce").dropna()
    after = pd.to_numeric(holdout[col], errors="coerce").dropna()
    if len(before) and len(after):
        shift_rows.append({
            "feature": col,
            "train_median": before.median(),
            "holdout_median": after.median(),
            "train_missing_pct": train[col].isna().mean() * 100,
            "holdout_missing_pct": holdout[col].isna().mean() * 100,
            "wasserstein_distance": wasserstein_distance(before, after),
        })
shift = pd.DataFrame(shift_rows).sort_values("wasserstein_distance", ascending=False)
shift.to_csv(REPORT_DIR / "feature_shift.csv", index=False)
display(shift)

## Save the review shortlist

The shortlist is intended for a human reviewer to examine against source records and procurement context. It should not be treated as a list of confirmed irregularities.

In [ ]:
top_50 = scores.head(50).copy()
top_50.to_csv(REPORT_DIR / "top_50_for_review.csv", index=False)
print("Saved the review shortlist and evaluation tables in reports/.")